# Distance between the nucleus centroid and the basal actin plane

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
# Minimum number of actin-fiber voxels in a z-slice for it to count as the actin plane
ACTIN_PLANE_MIN_VOXELS = 1000


def parse_file_name(filename, replicate):
    """
    Read date, adherence time, condition, pillar pitch, stack and nucleus number from a
    coordinate file name `<date>_HeLa_HP1aGFP_REP<n>_<time>_<height>_<pitch>_Z_stack<s>_nuc<k>_...`,
    where `<pitch>` is `2umpitch`, `3umpitch`, `4umpitch` or `control`.
    """

    parts = filename.split("_")
    date, time, condition = parts[0], parts[4], parts[5]
    pillar_step = parts[6].replace("umpitch", "um")

    return {
        "Date": date,
        "Replicate": replicate,
        "Adherence_Time": time,
        "condition": condition,
        "pillar_step": pillar_step,
        "Z_Stack": re.search(r"stack\d+", filename).group(0),
        "nucleus_nr": re.search(r"nuc\d+", filename).group(0),
    }


def actin_nucleus_distances(nucleus_file, actin_file):
    """
    Return the z measurements (µm) of one nucleus.

    The actin plane is the lowest z-slice with more than `ACTIN_PLANE_MIN_VOXELS`
    actin-fiber voxels, which marks the basal cell membrane.
    `Distance_nuc_centroid_actin_plane` is the height of the nucleus centroid above this
    plane, NaN if no slice has enough actin.

    Parameters
    ----------
    nucleus_file : Path
        Physical coordinates of the nucleus (`.npy`, columns z, y, x).
    actin_file : Path
        Physical coordinates of the actin fibers of the cell (`.npz`).
    """

    nucleus = np.load(nucleus_file)
    nuc_centroid_z = np.mean(nucleus, axis=0)[0]
    nuc_z_min = np.min(nucleus[:, 0])
    nuc_z_max = np.max(nucleus[:, 0])

    # All actin-fiber voxels of the cell, and the number of voxels in each z-slice
    actin = np.load(actin_file)
    all_fibers = np.vstack([actin[key] for key in actin.files])
    slice_z, voxels_per_slice = np.unique(all_fibers[:, 0], return_counts=True)

    dense_slices = np.where(voxels_per_slice > ACTIN_PLANE_MIN_VOXELS)[0]
    if dense_slices.size:
        actin_plane_z = slice_z[dense_slices[0]]  # the lowest such slice
        distance = nuc_centroid_z - actin_plane_z
    else:
        actin_plane_z = distance = np.nan

    return {
        "Nuc_Z_Min": nuc_z_min,
        "Nuc_Z_Max": nuc_z_max,
        "Nuc_Height": nuc_z_max - nuc_z_min,
        "Actin_Plane_Z": actin_plane_z,
        "Distance_nuc_centroid_actin_plane": distance,
    }

## Settings

Paths are relative to this notebook.

In [ ]:
DATA = Path("../../data")

# Coordinate folder of each replicate
REPLICATES = {rep: DATA / rep / "coordinate_files" for rep in ("REP1", "REP2", "REP3")}
OUTPUT = Path("nucleus_actin_distances_z_profile_REP1_REP2_REP3.csv")

## Compute the distances

In [ ]:
rows = []
for replicate, coordinate_dir in REPLICATES.items():
    coordinate_dir = Path(coordinate_dir)
    actin_files = sorted((coordinate_dir / "actin_fibers").rglob("*_actin_fibers_physical_coords.npz"))
    print(f"{replicate}: {len(actin_files)} actin files")

    for actin_file in actin_files:
        # Same <time>/<height>/<pitch>/ subfolder in nuclei/ as in actin_fibers/
        nucleus_file = (
            coordinate_dir
            / "nuclei"
            / actin_file.parent.relative_to(coordinate_dir / "actin_fibers")
            / actin_file.name.replace("_actin_fibers_physical_coords.npz", "_physical_coords.npy")
        )
        if not nucleus_file.exists():
            print(f"  No nucleus file for {actin_file.name}, skipped")
            continue
        rows.append(
            {**parse_file_name(actin_file.name, replicate), **actin_nucleus_distances(nucleus_file, actin_file)}
        )

df = pd.DataFrame(rows)
df.to_csv(OUTPUT, sep="\t")
print(f"Wrote {len(df)} nuclei to {OUTPUT}")
df.head()